In [1]:
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional
import json
import re
import statistics

from tqdm.auto import tqdm
from transformers import AutoTokenizer


SPIDER_ROOT = Path("benchmarks_2") / "spider_data"
FORMAT_ROOT = SPIDER_ROOT / "format_data"
MODEL_NAME = "Qwen/Qwen3-0.6B"

# Match scripts/qwen/sft/sft_qwen3_0.6b.sh and related Qwen KD scripts.
QWEN_BASE_PROCESS_CONFIG = {
    "name": "qwen_base",
    "data_dir": FORMAT_ROOT,
    "processed_dir": Path("processed_data") / "benchmarks" / "spider_data" / "qwen",
    "model_name": MODEL_NAME,
    "max_length": 1612,
    "max_prompt_length": 1479,
    "t_max_length": 2048,
    "t_max_prompt_length": 1800,
}

# Match scripts/qwen/synid_ce_*/train_0.6b_4b_*.sh.
QWEN_SYNID_PROCESS_CONFIG = {
    **QWEN_BASE_PROCESS_CONFIG,
    "max_length": 2048,
    "max_prompt_length": 1536,
    "t_max_length": 2048,
    "t_max_prompt_length": 1800,
}

# Add more runs here when you want to compare other processed outputs.
PROCESS_RUNS = [
    QWEN_BASE_PROCESS_CONFIG,
    {
        **QWEN_SYNID_PROCESS_CONFIG,
        "name": "qwen_synid_privileged",
        "data_dir": FORMAT_ROOT / "synid_aug_v2",
        "processed_dir": Path("processed_data") / "benchmarks" / "spider_data" / "synid_privileged" / "qwen",
    },
    {
        **QWEN_SYNID_PROCESS_CONFIG,
        "name": "qwen_synid_privileged_lora",
        "data_dir": FORMAT_ROOT / "synid_aug_v2_lora",
        "processed_dir": Path("processed_data") / "benchmarks" / "spider_data" / "synid_privileged_lora" / "qwen",
    },
    {
        **QWEN_BASE_PROCESS_CONFIG,
        "name": "llama_base_existing_rows_only",
        "processed_dir": Path("processed_data") / "spider_data" / "llama",
        "rows_only": True,
    },
]


PROMPT_FIELD_CONFIGS = {
    "student": {"system_field": "system_prompt", "user_field": "user_prompt"},
    "teacher": {"system_field": "t_system_prompt", "user_field": "t_user_prompt"},
}

PROCESS_SPLIT_FILES = {
    "train": "train.jsonl",
    "valid": "dev.jsonl",
    "test": "test.jsonl",
}


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as file:
        for line_no, line in enumerate(file, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            if not isinstance(row, dict):
                raise ValueError(f"{path}:{line_no} is not a JSON object")
            rows.append(row)
    return rows


def jsonl_count(path: Path) -> Optional[int]:
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as file:
        return sum(1 for line in file if line.strip())


d:\GitHub\text2sql_distillation_draft\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def discover_raw_spider_files(root: Path) -> List[Dict[str, Any]]:
    rows = []
    for path in sorted(root.glob("*.json")):
        try:
            data = read_json(path)
        except json.JSONDecodeError:
            continue

        count = len(data) if isinstance(data, list) else None
        rows.append(
            {
                "file": str(path),
                "kind": type(data).__name__,
                "rows": count,
                "keys": sorted(data[0].keys()) if isinstance(data, list) and data and isinstance(data[0], dict) else [],
            }
        )
    return rows


def discover_prompt_files(format_root: Path) -> List[Dict[str, Any]]:
    files = []
    for path in sorted(format_root.rglob("*.jsonl")):
        rel = path.relative_to(format_root)
        dataset = "base" if len(rel.parts) == 1 else "/".join(rel.parts[:-1])
        split_or_file = path.stem
        rows = read_jsonl(path)
        first = rows[0] if rows else {}
        prompt_sources = [
            prompt_name
            for prompt_name, fields in PROMPT_FIELD_CONFIGS.items()
            if all(field in first for field in fields.values())
        ]
        files.append(
            {
                "dataset": dataset,
                "split_or_file": split_or_file,
                "path": path,
                "rows": len(rows),
                "keys": sorted(first.keys()),
                "prompt_sources": prompt_sources,
                "has_response": "response" in first,
            }
        )
    return files


raw_spider_files = discover_raw_spider_files(SPIDER_ROOT)
prompt_files = discover_prompt_files(FORMAT_ROOT)

{
    "spider_root": str(SPIDER_ROOT),
    "raw_json_files": raw_spider_files,
    "prompt_jsonl_files": [
        {**item, "path": str(item["path"])}
        for item in prompt_files
    ],
}


{'spider_root': 'benchmarks_2\\spider_data',
 'raw_json_files': [{'file': 'benchmarks_2\\spider_data\\dev.json',
   'kind': 'list',
   'rows': 1034,
   'keys': ['db_id',
    'query',
    'query_toks',
    'query_toks_no_value',
    'question',
    'question_toks',
    'sql']},
  {'file': 'benchmarks_2\\spider_data\\tables.json',
   'kind': 'list',
   'rows': 166,
   'keys': ['column_names',
    'column_names_original',
    'column_types',
    'db_id',
    'foreign_keys',
    'primary_keys',
    'table_names',
    'table_names_original']},
  {'file': 'benchmarks_2\\spider_data\\test.json',
   'kind': 'list',
   'rows': 2147,
   'keys': ['db_id',
    'query',
    'query_toks',
    'query_toks_no_value',
    'question',
    'question_toks',
    'sql']},
  {'file': 'benchmarks_2\\spider_data\\test_tables.json',
   'kind': 'list',
   'rows': 206,
   'keys': ['column_names',
    'column_names_original',
    'column_types',
    'db_id',
    'foreign_keys',
    'primary_keys',
    'table_names

In [3]:
def apply_chat_template(tokenizer, system_prompt: str, user_prompt: str, enable_thinking: bool = False) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def encode_len(text: str) -> int:
    # process_data.py uses encode(..., add_special_tokens=False); keep this identical.
    return len(tokenizer.encode(text, add_special_tokens=False))


def response_token_len(prompt_text: str, prompt_raw_len: int, response_text: str) -> int:
    # process_data.py appends EOS to train/valid responses.
    eos = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
    full_tokens = tokenizer.encode(prompt_text + response_text, add_special_tokens=False) + eos
    return len(full_tokens[prompt_raw_len:])


def summarize_lengths(lengths: List[int]) -> Dict[str, Any]:
    if not lengths:
        return {}

    values = sorted(lengths)
    n = len(values)

    def percentile(p: float) -> float:
        if n == 1:
            return float(values[0])
        idx = (n - 1) * p
        low = int(idx)
        high = min(low + 1, n - 1)
        frac = idx - low
        return values[low] * (1 - frac) + values[high] * frac

    return {
        "count": n,
        "min": min(values),
        "max": max(values),
        "mean": statistics.mean(values),
        "median": statistics.median(values),
        "p90": percentile(0.90),
        "p95": percentile(0.95),
        "p99": percentile(0.99),
    }


def extract_prompt_block(user_prompt: str, header: str) -> str:
    marker = re.search(rf"(?im)^\s*{re.escape(header)}:\s*\n", user_prompt)
    if marker is None:
        return ""

    start = marker.end()
    next_header = re.search(
        r"(?im)^\s*(QUESTION|PROBLEM|EVIDENCE|SCHEMA|DATABASE SCHEMA|HERE IS A REFERENCE SOLUTION|ANSWER):\s*$",
        user_prompt[start:],
    )
    end = start + next_header.start() if next_header else len(user_prompt)
    return user_prompt[start:end].strip()


def prompt_fields_available(row: Dict[str, Any], prompt_source: str) -> bool:
    fields = PROMPT_FIELD_CONFIGS[prompt_source]
    return all(field in row for field in fields.values())


In [4]:
def analyze_prompt_rows(
    rows: List[Dict[str, Any]],
    *,
    dataset: str,
    split_or_file: str,
    source_path: Path,
    prompt_source: str,
    enable_thinking: bool = False,
) -> List[Dict[str, Any]]:
    fields = PROMPT_FIELD_CONFIGS[prompt_source]
    analyzed = []

    for idx, item in tqdm(
        enumerate(rows),
        total=len(rows),
        desc=f"{dataset}/{split_or_file}/{prompt_source}",
    ):
        prompt_text = apply_chat_template(
            tokenizer,
            system_prompt=item[fields["system_field"]],
            user_prompt=item[fields["user_field"]],
            enable_thinking=enable_thinking,
        )
        prompt_len = encode_len(prompt_text)
        response_text = str(item.get("response", ""))
        response_len = response_token_len(prompt_text, prompt_len, response_text) if "response" in item else 0

        user_prompt = str(item[fields["user_field"]])
        analyzed.append(
            {
                "dataset": dataset,
                "split_or_file": split_or_file,
                "source_path": str(source_path),
                "idx": idx,
                "prompt_source": prompt_source,
                "question": (
                    extract_prompt_block(user_prompt, "QUESTION")
                    or extract_prompt_block(user_prompt, "Question")
                    or extract_prompt_block(user_prompt, "Problem")
                ),
                "schema_chars": len(
                    extract_prompt_block(user_prompt, "DATABASE SCHEMA")
                    or extract_prompt_block(user_prompt, "SCHEMA")
                    or extract_prompt_block(user_prompt, "Database Schema")
                ),
                "prompt_len": prompt_len,
                "response_len": response_len,
                "prompt_plus_response_len": prompt_len + response_len,
            }
        )

    return analyzed


all_token_rows = []
for file_info in prompt_files:
    rows = read_jsonl(file_info["path"])
    for prompt_source in file_info["prompt_sources"]:
        all_token_rows.extend(
            analyze_prompt_rows(
                rows,
                dataset=file_info["dataset"],
                split_or_file=file_info["split_or_file"],
                source_path=file_info["path"],
                prompt_source=prompt_source,
            )
        )

summary_rows = []
for key in sorted({(r["dataset"], r["split_or_file"], r["prompt_source"]) for r in all_token_rows}):
    dataset, split_or_file, prompt_source = key
    subset = [
        row
        for row in all_token_rows
        if (row["dataset"], row["split_or_file"], row["prompt_source"]) == key
    ]
    prompt_summary = summarize_lengths([row["prompt_len"] for row in subset])
    response_summary = summarize_lengths([row["response_len"] for row in subset])
    full_summary = summarize_lengths([row["prompt_plus_response_len"] for row in subset])
    summary_rows.append(
        {
            "dataset": dataset,
            "split_or_file": split_or_file,
            "prompt_source": prompt_source,
            "rows": len(subset),
            "prompt_max": prompt_summary.get("max"),
            "prompt_p95": prompt_summary.get("p95"),
            "prompt_p99": prompt_summary.get("p99"),
            "response_max": response_summary.get("max"),
            "full_max": full_summary.get("max"),
            "full_p95": full_summary.get("p95"),
            "full_p99": full_summary.get("p99"),
        }
    )

try:
    import pandas as pd

    token_summary = pd.DataFrame(summary_rows)
    token_summary
except ImportError:
    summary_rows


base/train/student: 100%|██████████| 7000/7000 [00:19<00:00, 367.39it/s]


In [5]:
def load_process_split_rows(data_dir: Path, process_split: str) -> tuple[Optional[Path], List[Dict[str, Any]]]:
    filename = PROCESS_SPLIT_FILES[process_split]
    path = data_dir / filename
    if not path.exists():
        return path, []
    return path, read_jsonl(path)


def merge_teacher_rows(
    student_rows: List[Dict[str, Any]],
    teacher_path: Path,
) -> tuple[List[Dict[str, Any]], List[str]]:
    issues = []
    if not teacher_path.exists():
        return student_rows, issues

    teacher_rows = read_jsonl(teacher_path)
    if len(student_rows) != len(teacher_rows):
        issues.append(f"teacher row mismatch: student={len(student_rows)} teacher={len(teacher_rows)}")
        return student_rows, issues

    merged_rows = []
    for idx, (student, teacher) in enumerate(zip(student_rows, teacher_rows)):
        missing = [field for field in ("t_system_prompt", "t_user_prompt", "response") if field not in teacher]
        if missing:
            issues.append(f"teacher row {idx} missing {missing}")
            continue
        if student.get("response") != teacher["response"]:
            issues.append(f"response mismatch at train row {idx}")
            continue
        merged = dict(student)
        merged["t_system_prompt"] = teacher["t_system_prompt"]
        merged["t_user_prompt"] = teacher["t_user_prompt"]
        merged_rows.append(merged)

    return merged_rows, issues


def simulate_encoded_row(row: Dict[str, Any], process_split: str, config: Dict[str, Any]) -> Dict[str, Any]:
    prompt_text = apply_chat_template(tokenizer, row["system_prompt"], row["user_prompt"], enable_thinking=False)
    prompt_raw_len = encode_len(prompt_text)
    prompt_len = min(prompt_raw_len, config["max_prompt_length"])
    prompt_truncated = prompt_raw_len > config["max_prompt_length"]

    response_len = 0
    response_truncated = False
    teacher_prompt_raw_len = None
    teacher_prompt_len = None
    teacher_prompt_truncated = False

    keep = True
    drop_reason = ""

    if process_split in ("train", "valid"):
        response_text = str(row.get("response", ""))
        response_len = response_token_len(prompt_text, prompt_raw_len, response_text)

        if process_split == "train" and prompt_fields_available(row, "teacher"):
            t_prompt_text = apply_chat_template(
                tokenizer,
                row["t_system_prompt"],
                row["t_user_prompt"],
                enable_thinking=False,
            )
            teacher_prompt_raw_len = encode_len(t_prompt_text)
            teacher_prompt_len = min(teacher_prompt_raw_len, config["t_max_prompt_length"])
            teacher_prompt_truncated = teacher_prompt_raw_len > config["t_max_prompt_length"]
            max_response_len = min(
                config["max_length"] - prompt_len,
                config["t_max_length"] - teacher_prompt_len,
            )
            if max_response_len <= 0:
                keep = False
                drop_reason = "no room for response after prompt truncation"
                response_len = 0
            elif response_len > max_response_len:
                response_len = max_response_len
                response_truncated = True
    else:
        # process_data.py checks test prompt length after max_prompt_length truncation.
        if prompt_len >= config["max_length"]:
            keep = False
            drop_reason = "test prompt >= max_length"

    return {
        "keep": keep,
        "drop_reason": drop_reason,
        "prompt_raw_len": prompt_raw_len,
        "prompt_len": prompt_len,
        "prompt_truncated": prompt_truncated,
        "response_len": response_len,
        "response_truncated": response_truncated,
        "sequence_len_written": prompt_len + (1 + response_len if process_split in ("train", "valid") and keep else 0),
        "teacher_prompt_raw_len": teacher_prompt_raw_len,
        "teacher_prompt_len": teacher_prompt_len,
        "teacher_prompt_truncated": teacher_prompt_truncated,
    }


def compare_process_run(config: Dict[str, Any]) -> List[Dict[str, Any]]:
    compare_rows = []
    data_dir = Path(config["data_dir"])
    processed_dir = Path(config["processed_dir"])
    teacher_path = data_dir / "teacher_train.jsonl"
    rows_only = bool(config.get("rows_only"))

    for process_split in ("train", "valid", "test"):
        input_path, rows = load_process_split_rows(data_dir, process_split)
        merge_issues = []
        if rows and process_split == "train" and teacher_path.exists():
            rows, merge_issues = merge_teacher_rows(rows, teacher_path)

        processed_jsonl = processed_dir / f"{process_split}.jsonl"
        processed_count = jsonl_count(processed_jsonl)

        if rows_only:
            expected_rows = len(rows) if input_path and input_path.exists() else None
            compare_rows.append(
                {
                    "run": config["name"],
                    "process_split": process_split,
                    "input_file": str(input_path) if input_path else None,
                    "input_exists": bool(input_path and input_path.exists()),
                    "input_rows_after_teacher_merge": len(rows) if input_path and input_path.exists() else None,
                    "expected_processed_rows": expected_rows,
                    "actual_processed_rows": processed_count,
                    "row_count_ok": processed_count == expected_rows if expected_rows is not None and processed_count is not None else None,
                    "student_prompt_truncated": None,
                    "teacher_prompt_truncated": None,
                    "response_truncated": None,
                    "dropped": None,
                    "max_prompt_raw_len": None,
                    "max_prompt_written_len": None,
                    "max_sequence_written_len": None,
                    "max_teacher_prompt_raw_len": None,
                    "merge_issues": "; ".join(merge_issues[:3]),
                }
            )
            continue

        encoded = [
            simulate_encoded_row(row, process_split, config)
            for row in tqdm(rows, desc=f"simulate {config['name']}/{process_split}")
        ]
        kept = [row for row in encoded if row["keep"]]

        compare_rows.append(
            {
                "run": config["name"],
                "process_split": process_split,
                "input_file": str(input_path) if input_path else None,
                "input_exists": bool(input_path and input_path.exists()),
                "input_rows_after_teacher_merge": len(rows),
                "expected_processed_rows": len(kept),
                "actual_processed_rows": processed_count,
                "row_count_ok": processed_count == len(kept) if processed_count is not None else None,
                "student_prompt_truncated": sum(row["prompt_truncated"] for row in kept),
                "teacher_prompt_truncated": sum(row["teacher_prompt_truncated"] for row in kept),
                "response_truncated": sum(row["response_truncated"] for row in kept),
                "dropped": len(encoded) - len(kept),
                "max_prompt_raw_len": max((row["prompt_raw_len"] for row in encoded), default=None),
                "max_prompt_written_len": max((row["prompt_len"] for row in kept), default=None),
                "max_sequence_written_len": max((row["sequence_len_written"] for row in kept), default=None),
                "max_teacher_prompt_raw_len": max(
                    (row["teacher_prompt_raw_len"] for row in kept if row["teacher_prompt_raw_len"] is not None),
                    default=None,
                ),
                "merge_issues": "; ".join(merge_issues[:3]),
            }
        )

    return compare_rows


In [6]:
process_compare_rows = []
for process_config in PROCESS_RUNS:
    process_compare_rows.extend(compare_process_run(process_config))

try:
    import pandas as pd

    process_compare = pd.DataFrame(process_compare_rows)
    process_compare
except ImportError:
    process_compare_rows


simulate qwen_synid_privileged/train: 100%|██████████| 7000/7000 [00:36<00:00, 193.32it/s]
simulate qwen_synid_privileged/valid: 0it [00:00, ?it/s]
simulate qwen_synid_privileged/test: 0it [00:00, ?it/s]
simulate qwen_synid_privileged_lora/train: 100%|██████████| 7000/7000 [01:18<00:00, 89.60it/s] 
simulate qwen_synid_privileged_lora/valid: 0it [00:00, ?it/s]
simulate qwen_synid_privileged_lora/test: 0it [00:00, ?it/s]


In [7]:
# Longest examples across all discovered prompt JSONL files.
top_n = 20
longest_context_rows = sorted(
    all_token_rows,
    key=lambda row: row["prompt_plus_response_len"],
    reverse=True,
)[:top_n]

try:
    import pandas as pd

    pd.DataFrame(longest_context_rows)[
        [
            "dataset",
            "split_or_file",
            "prompt_source",
            "idx",
            "prompt_len",
            "response_len",
            "prompt_plus_response_len",
            "schema_chars",
            "question",
            "source_path",
        ]
    ]
except ImportError:
    longest_context_rows


In [8]:
# Rows that exceed the active Qwen3-0.6B processing thresholds.
def limits_for_token_row(row: Dict[str, Any]) -> Dict[str, int]:
    if row["dataset"] == "base":
        config = QWEN_BASE_PROCESS_CONFIG
    else:
        config = QWEN_SYNID_PROCESS_CONFIG

    if row["prompt_source"] == "teacher":
        return {
            "prompt_limit": config["t_max_prompt_length"],
            "full_limit": config["t_max_length"],
        }
    return {
        "prompt_limit": config["max_prompt_length"],
        "full_limit": config["max_length"],
    }


threshold_rows = []
for row in all_token_rows:
    limits = limits_for_token_row(row)
    exceeds_prompt = row["prompt_len"] > limits["prompt_limit"]
    exceeds_full = row["prompt_plus_response_len"] > limits["full_limit"]
    if exceeds_prompt or exceeds_full:
        threshold_rows.append(
            {
                **row,
                **limits,
                "exceeds_prompt_limit": exceeds_prompt,
                "exceeds_full_limit": exceeds_full,
            }
        )

try:
    import pandas as pd

    pd.DataFrame(threshold_rows)[
        [
            "dataset",
            "split_or_file",
            "prompt_source",
            "idx",
            "prompt_len",
            "response_len",
            "prompt_plus_response_len",
            "prompt_limit",
            "full_limit",
            "exceeds_prompt_limit",
            "exceeds_full_limit",
            "question",
            "source_path",
        ]
    ]
except ImportError:
    threshold_rows
